# Gold Price Regression

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `gold_price_usd`

## 2 · Project Overview

We predict **gold price** (USD per ounce) based on macro-economic indicators — USD index, Treasury yields, inflation, VIX, oil prices, silver prices, and Fed funds rate.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given daily market indicators (USD index, bond yields, inflation rate, VIX, oil price, silver price, Fed funds rate, S&P 500), predict the gold price in USD per ounce.

## 5 · Why This Project Matters

- **Gold price prediction** is central to commodity trading and portfolio hedging.
- Understanding gold's macro drivers enables smarter asset allocation.
- Gold serves as an inflation hedge and safe-haven asset.
- Demonstrates regression with correlated financial features.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 6,500 |
| **Features** | sp500, dxy_index, us_10y_yield, crude_oil, inflation_rate, vix, silver_price, fed_funds_rate |
| **Target** | gold_price_usd (continuous, USD/oz) |
| **Range** | ~$800 – $3,000 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "gold_price_usd"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

Synthetic dataset: 6,500 daily observations of gold price with economic and market indicators.

In [ ]:
np.random.seed(SEED)
n = 6500
# Simulate daily market data
days = np.arange(n)
trend = 1200 + 0.15 * days  # slow upward trend over ~18 years

sp500 = np.round(2000 + 0.5 * days + 200 * np.sin(2 * np.pi * days / 365)
                 + np.cumsum(np.random.normal(0, 5, n)), 2).clip(1000, 6000)
dxy_index = np.round(95 - 0.002 * days + 3 * np.sin(2 * np.pi * days / 730)
                     + np.cumsum(np.random.normal(0, 0.1, n)), 2).clip(70, 120)
us_10y_yield = np.round(2.5 - 0.0002 * days + 0.5 * np.sin(2 * np.pi * days / 500)
                        + np.cumsum(np.random.normal(0, 0.02, n)), 2).clip(0.5, 5.0)
crude_oil = np.round(55 + 0.005 * days + 10 * np.sin(2 * np.pi * days / 400)
                     + np.cumsum(np.random.normal(0, 1, n)), 2).clip(20, 150)
inflation_rate = np.round(2.0 + 0.5 * np.sin(2 * np.pi * days / 600)
                          + np.random.normal(0, 0.3, n), 2).clip(0, 8)
vix = np.round(18 + 5 * np.sin(2 * np.pi * days / 300)
               + np.random.exponential(3, n), 2).clip(10, 80)
silver_price = np.round(15 + 0.003 * days + np.cumsum(np.random.normal(0, 0.1, n)), 2).clip(10, 50)
fed_funds_rate = np.round(1.5 + np.cumsum(np.random.normal(0, 0.005, n)), 2).clip(0, 5.5)

# Gold price model: trend + inverse USD + inflation hedge + fear premium
gold_price_usd = (trend
                  - 8 * (dxy_index - 95)
                  + 50 * (inflation_rate - 2)
                  + 3 * vix
                  - 30 * (us_10y_yield - 2.5)
                  + 5 * silver_price
                  + 0.01 * crude_oil)
gold_price_usd = np.round(gold_price_usd + np.random.normal(0, 30, n), 2).clip(800, 3000)

df = pd.DataFrame({
    "sp500": sp500, "dxy_index": dxy_index, "us_10y_yield": us_10y_yield,
    "crude_oil": crude_oil, "inflation_rate": inflation_rate, "vix": vix,
    "silver_price": silver_price, "fed_funds_rate": fed_funds_rate,
    "gold_price_usd": gold_price_usd
})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['gold_price_usd'].describe()}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["dxy_index"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("USD Index (DXY)"); axes[0][0].set_ylabel("Gold ($)")
axes[0][0].set_title("USD Index vs Gold")

axes[0][1].scatter(df["inflation_rate"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("Inflation Rate (%)"); axes[0][1].set_ylabel("Gold ($)")
axes[0][1].set_title("Inflation vs Gold")

axes[0][2].scatter(df["vix"], df[TARGET], alpha=0.2, s=10)
axes[0][2].set_xlabel("VIX"); axes[0][2].set_ylabel("Gold ($)")
axes[0][2].set_title("VIX vs Gold")

axes[1][0].scatter(df["us_10y_yield"], df[TARGET], alpha=0.2, s=10)
axes[1][0].set_xlabel("10Y Treasury Yield"); axes[1][0].set_ylabel("Gold ($)")
axes[1][0].set_title("10Y Yield vs Gold")

axes[1][1].scatter(df["silver_price"], df[TARGET], alpha=0.2, s=10)
axes[1][1].set_xlabel("Silver Price ($)"); axes[1][1].set_ylabel("Gold ($)")
axes[1][1].set_title("Silver vs Gold")

corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink": 0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `gold_price_usd`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="goldenrod", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Gold Price (USD/oz)")

axes[1].plot(df[TARGET].values, color="goldenrod", alpha=0.7, lw=0.5)
axes[1].set_title("Gold Price — Time Series View")
axes[1].set_xlabel("Day Index"); axes[1].set_ylabel("Gold Price ($)")

plt.tight_layout()
plt.show()
print(f"Range: ${df[TARGET].min():,.2f} to ${df[TARGET].max():,.2f}")
print(f"Mean: ${df[TARGET].mean():,.2f}, Median: ${df[TARGET].median():,.2f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: None (all numeric features).
- **Scaling**: Not needed for tree models.
- **Note**: This is financial cross-sectional regression, not time-series forecasting. No temporal leakage since we use random split.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["gold_silver_ratio_proxy"] = X_train["silver_price"]  # direct proxy
X_test["gold_silver_ratio_proxy"] = X_test["silver_price"]

X_train["real_yield"] = X_train["us_10y_yield"] - X_train["inflation_rate"]
X_test["real_yield"] = X_test["us_10y_yield"] - X_test["inflation_rate"]

X_train["fear_gauge"] = X_train["vix"] * X_train["inflation_rate"]
X_test["fear_gauge"] = X_test["vix"] * X_test["inflation_rate"]

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **USD Index (DXY)** has the strongest inverse relationship with gold.
- **Inflation** drives gold upward (inflation hedge thesis confirmed).
- **VIX** (fear gauge) increases gold demand during uncertainty.
- **Silver price** is highly correlated (precious metals move together).
- **10Y Treasury yield** inversely affects gold (opportunity cost).
- **Real yield** (yield - inflation) is the most informative engineered feature.

**Business takeaway:** Gold is anti-dollar, pro-inflation, and fear-driven. Real yields are the best single indicator.

## 26 · Limitations

1. Synthetic data with simplified financial relationships.
2. Random split — not proper time-series backtesting.
3. Missing geopolitical event features.
4. No central bank gold reserves data.
5. Simplified linear macro relationships.

## 27 · How to Improve This Project

1. Use real historical gold price and macro data.
2. Apply proper time-series walk-forward validation.
3. Add geopolitical risk indices.
4. Include gold ETF flows and futures positioning.
5. Model regime changes (risk-on vs. risk-off).

## 28 · Production Considerations

- Deploy for gold price scenario analysis.
- Integrate with portfolio optimization tools.
- Monitor for regime changes (correlation breaks).
- Update daily with market data feeds.
- Output prediction intervals for trading decisions.

## 29 · Common Mistakes

1. Using random train/test split for financial data (temporal leakage risk).
2. Not computing real yields (yield - inflation).
3. Ignoring the USD-gold inverse relationship.
4. Treating VIX as noise rather than a flight-to-safety signal.
5. Not separating trend from cyclical components.

## 30 · Mini Challenge / Exercises

1. Compute `real_yield = us_10y_yield - inflation_rate` — is it more predictive than either alone?
2. Plot gold vs. DXY — confirm the inverse relationship.
3. Remove `silver_price` (highly correlated) — does R² drop much?
4. Sort by index and use last 20% as test (time-aware split).
5. Calculate gold/silver ratio and analyze its predictive power.

## 31 · Final Summary / Key Takeaways

1. **USD Index** is the strongest inverse gold predictor.
2. **Inflation** drives gold as a hedge asset.
3. **VIX** captures flight-to-safety demand.
4. **Real yield** (yield - inflation) is the key engineered feature.
5. **Silver** co-moves strongly (precious metals correlation).
6. **Financial regression** requires careful treatment of temporal dependencies.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))